# Close-to-close vs Open-to-open — do our signals hold up?

`cta.Simulate` uses close-to-close (c2c) returns by default: `ret[t] = close[t]/close[t-1] - 1`. This captures {last night's session + today's day session}. But a lot of Taiwan-relevant information (US markets closing overnight, Fed announcements, etc.) shows up in the **overnight gap** — so measuring PnL on {today's day session + tonight's session}, i.e. open-to-open (`ret_o2o[t] = open[t]/open[t-1] - 1`), might tell a different story.

## Timing convention (same shift as `cta.Simulate`)

For both return modes, we use `exec_sig = signal.shift(2)`:

| return mode | pnl[t] | hold period | when signal known |
|---|---|---|---|
| c2c (default)  | `signal[t-2] × (close[t]/close[t-1] - 1)` | `close[t-1] → close[t]` | before close[t-2] |
| **o2o (this notebook)** | `signal[t-2] × (open[t]/open[t-1] - 1)` | `open[t-1] → open[t]`  | before close[t-2] |

Same 2-bar decision lag → **apples-to-apples comparison**. The only difference is which 24-hour window each measures:
- **c2c** captures last night's overnight session + today's day session
- **o2o** captures today's day session + tonight's overnight session

## What to look for

- Signals with high SR under **o2o but not c2c** = signal predicts overnight moves (US-driven, macro, positioning-flow anticipating overnight)
- Signals with high SR under **c2c but not o2o** = signal predicts intraday day-session moves (local positioning, TW-domestic flows)
- The combined portfolio should shift too — some solos may deserve re-weighting.

## MTX return diagnostics (2013+)

| metric | c2c | o2o | overnight gap | day move (o→c) |
|---|---|---|---|---|
| mean | +0.059% | +0.059% | +0.049% | +0.009% |
| std  | 1.164% | 1.125% | 0.843% | 0.772% |
| skew | −0.64  | −0.74  | −0.65  | −0.48  |

Overnight gap contributes MORE mean AND MORE variance than the day session — so overnight-signal edges should show more prominently in o2o.

**c2c ↔ o2o correlation: only +0.52** — they're genuinely different exposures despite same underlying prices.


In [ ]:
import importlib, sys, warnings, csv
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

warnings.filterwarnings("ignore", message="findfont: .* not found")
_inst = {f.name for f in fm.fontManager.ttflist}
_cand = ["Hiragino Sans GB","Heiti TC","Songti SC","PingFang HK","PingFang SC","PingFang TC",
         "Arial Unicode MS","Noto Sans CJK TC","DejaVu Sans"]
mpl.rcParams["font.family"] = [f for f in _cand if f in _inst] or ["DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

import cta
for _m in ["cta.asset","cta.operators","cta.simulate","cta.simulate_dollars",
          "cta.large_trader","cta.options","cta.three_majors","cta.tsmc_events",
          "cta.us_indexes","cta.signal_stats"]:
    if _m in sys.modules: importlib.reload(sys.modules[_m])
importlib.reload(cta)

EVAL_START, EVAL_END = "2013-01-01", "2026-07-27"

ASSET = cta.load_asset("mtx","1d"); cta.set_active_asset(ASSET)
close = ASSET["close"].astype(float)
open_ = ASSET["open"].astype(float)
high  = ASSET["high"].astype(float);  low = ASSET["low"].astype(float)
back_close = ASSET["back_close"].astype(float)
volume = ASSET["volume"].astype(float) if "volume" in ASSET.columns else None

ret_c2c = close.pct_change()
ret_o2o = open_.pct_change()   # ret_o2o[t] = open[t]/open[t-1] - 1

# Cost — reference price is CLOSE for c2c, OPEN for o2o (turnover happens at
# the reference price of each mode)
cost_c2c = 20.0 / (close * 50.0) + 0.00002
cost_o2o = 20.0 / (open_ * 50.0) + 0.00002

print(f"asset: {ASSET.symbol} {ASSET.time_granularity} {len(ASSET):,} bars   ({ASSET.index[0].date()} → {ASSET.index[-1].date()})")
print(f"eval : {EVAL_START} → {EVAL_END}")
print(f"c2c ↔ o2o correlation (2013+): {ret_c2c.loc[EVAL_START:].corr(ret_o2o.loc[EVAL_START:]):+.3f}")


## 1. Rebuild the 9 survivors from the aggregate notebook

Same recipes — no re-search here, just apply o2o to the winners we already selected.


In [ ]:
def _bd(s): return np.tanh(s.replace([np.inf,-np.inf], np.nan))
def _dev(x,w): return x - cta.InstMean(w,x)
def _selfz(x,w):
    mu=cta.InstMean(w,x); sd=cta.InstStdev(w,x).replace(0,np.nan); return (x-mu)/sd
def _selfz_winsor(x,w,c=3.0): return _selfz(x,w).clip(-c,c)/c
def _robust_z(x,w):
    med=x.rolling(w,min_periods=max(3,w//2)).median()
    mad=(x-med).abs().rolling(w,min_periods=max(3,w//2)).median()
    return (x-med)/(1.4826*mad).replace(0,np.nan)
def _sign_thresh(x,w,t=0.5):
    z=_selfz(x,w); return pd.Series(np.where(z>t,1.0,np.where(z<-t,-1.0,0.0)),index=x.index)
def _rank_c(x,w): return (cta.InstRank(w,x)-0.5)*2

# ── Load raw sources ──────────────────────────────────────────────────────
tx_b10=cta.load_large_trader("TX","top10_buy").astype(float)
tx_s10=cta.load_large_trader("TX","top10_sell").astype(float)
tx_net10=cta.load_large_trader("TX","top10_net_pct").astype(float)
tx_log_ratio=np.log(tx_b10.replace(0,np.nan)/tx_s10.replace(0,np.nan))
put_mo=cta.load_option_daily_total("oi","put","monthly").astype(float)
call_all=cta.load_option_daily_total("oi","call","all").astype(float)
sox_c=cta.load_us_index_tw("^SOX",ASSET.index,"close"); spy_c=cta.load_us_index_tw("SPY",ASSET.index,"close")
_lr=np.log(sox_c/spy_c)
_r_sox_20=sox_c.pct_change(20); _r_spy_20=spy_c.pct_change(20)
_denom_20=_r_spy_20.where(_r_spy_20.abs()>0.003,np.nan)
_sox_amp20=(np.sign(_r_spy_20)*(_r_sox_20/_denom_20)).clip(-10,10)
_sox_r1=sox_c.pct_change(1); _spy_r1=spy_c.pct_change(1)
_rollcorr=_sox_r1.rolling(60,min_periods=20).corr(_spy_r1)
_ret_prod5=sox_c.pct_change(5)*spy_c.pct_change(5)

# ── Survivor recipes (exact same as best_signals_aggregate § 3-7) ─────────
RAW = {
    "LT_top10npct_signth_W60":         _sign_thresh(tx_net10,60),
    "LT_logratio_rankc_W20":           _rank_c(tx_log_ratio,20),
    "LT_logratio_selftanh_W20":        _bd(_selfz(tx_log_ratio,20)),
    "OPT_call_all_oi_signth_W20":      _sign_thresh(call_all,20),
    "OPT_put_mo_oi_selftanh_W60":      _bd(_selfz(put_mo,60)),
    "US_log_ratio_pct_chg_W120":       _lr - _lr.shift(120),
    "US_sox_agree_amp_N20_robustW20":  _robust_z(_sox_amp20,20),
    "US_sox_spy_rollcorr_W60_rz_W120": _robust_z(_rollcorr,120),
    "US_sox_spy_ret_product_N5_pctW20":_ret_prod5 - _ret_prod5.shift(20),
}
PRE = {n: s.replace([np.inf,-np.inf],np.nan).fillna(0.0) for n,s in RAW.items()}
NORM = {n: cta.normalize_signal(s, method="tanh", window=252, force=True).rename(n)
        for n,s in PRE.items()}

# Auto-flip on c2c (baseline) — keep the SAME signs for the o2o comparison
_tbl_flip = cta.batch_signal_stats(NORM, ASSET, start=EVAL_START, end=EVAL_END, auto_flip=True)
SIGNS = {n: int(_tbl_flip.loc[n,"sign"]) for n in _tbl_flip.index}
SIGNED = {n: (NORM[n] * SIGNS[n]).rename(n) for n in NORM}

print(f"survivors: {len(SIGNED)}")
print(f"signs (c2c-optimal, kept for o2o):")
for n, s in SIGNS.items():
    src = n.split("_")[0]; print(f"  {src:4s}  {n:38s}  sign={s:+d}")


## 0.5. Timing convention — explicit walkthrough

**Your intended operation order**:

| day | event |
|---|---|
| **t − 1** | signal computed (using data known at or before EOD t−1) |
| **t**     | trade at market **open** → position `p_t = signal[t−1]` established |
| **t + 1** | at open, **rebalance** to `p_{t+1} = signal[t]`; PnL of the hold period settles |

The hold period `open[t] → open[t+1]` earns return `r = open[t+1]/open[t] − 1`. Attributed to day **t+1** (the settlement date), the P&L is:

```
    pnl[t+1] = signal[t−1] × (open[t+1]/open[t] − 1)
```

Substituting `u = t + 1`:

```
    pnl[u] = signal[u−2] × (open[u]/open[u−1] − 1)
           = signal.shift(2)[u] × ret_o2o[u]
```

**This is exactly what the `pnl_of(sig, ret_o2o, cost_o2o)` function in §2 computes.**

**Turnover / cost** — at open[u], position changes from `p_{u−1} = signal[u−3]` to `p_u = signal[u−2]`. Cost attributed to day u:

```
    turnover[u] = |exec_sig[u] − exec_sig[u−1]| = |signal[u−2] − signal[u−3]|
    cost[u]     = turnover[u] × cost_pct[u]
```

The cost formula uses `open[u]` in the denominator (since the trade happens at `open[u]`, not at `close[u]`):

```
    cost_pct[u] = fixed_per_side / (open[u] × point_value) + fee_rate
```

The next cell traces a specific date through the arithmetic to confirm the formula.


In [ ]:
# ── § 0.5.1  Concrete arithmetic trace (verifies the timing spec above) ─────

# Use a linear-ramp signal (signal[t] = t) so the arithmetic is easy to eyeball.
_sig_trace = pd.Series(np.arange(len(ASSET)), index=ASSET.index, dtype=float)

def pnl_of(sig, ret, cost=None):
    """PnL under the o2o timing convention (§ 0.5).

    exec_sig = signal.shift(2)     # signal[u-2] drives pnl[u]
    pnl[u]   = exec_sig[u] × ret[u]  — u = "settlement day t+1"; hold = open[u-1]→open[u]
    cost[u]  = |exec_sig[u] - exec_sig[u-1]| × cost_pct[u]
    """
    ex = sig.shift(2)
    p  = ex * ret
    if cost is not None:
        p = p - ex.fillna(0).diff().abs() * cost
    return p

# Pick a date near the end of history
u_i   = len(ASSET) - 5
u     = ASSET.index[u_i]        # PnL settlement day (== t+1)
t     = ASSET.index[u_i - 1]    # Trade day
t_m1  = ASSET.index[u_i - 2]    # Signal day
t_p2  = ASSET.index[u_i + 1]    # Next PnL settlement day

pnl_trace = pnl_of(_sig_trace, ret_o2o)

print("=== Timing trace ===")
print(f"  t-1  {t_m1.date()}    signal computed → signal[t-1] = {_sig_trace.loc[t_m1]:.0f}")
print(f"  t    {t.date()}    trade at open  → open[t]   = {open_.loc[t]:,.1f}")
print(f"  t+1  {u.date()}    rebalance/PnL → open[t+1] = {open_.loc[u]:,.1f}")

_r = open_.loc[u]/open_.loc[t] - 1
_expected = _sig_trace.loc[t_m1] * _r
_actual   = pnl_trace.loc[u]

print(f"\n  ret o2o over the hold = open[t+1]/open[t] − 1 = {_r:+.6f}")
print(f"  expected pnl[t+1] = signal[t-1] × ret = {_sig_trace.loc[t_m1]:.0f} × {_r:+.6f} = {_expected:+.6f}")
print(f"  computed pnl[t+1] = pnl_of(signal, ret_o2o).loc[t+1]     = {_actual:+.6f}")
print(f"  {'✅ MATCH' if np.isclose(_actual, _expected) else '❌ MISMATCH — CHECK THE MATH'}")

# Also verify: the NEXT day uses signal[t], confirming rebalance-at-open[t+1] semantics
_r2   = open_.loc[t_p2]/open_.loc[u] - 1
_exp2 = _sig_trace.loc[t] * _r2
_act2 = pnl_trace.loc[t_p2]
print(f"\n  Rebalance check — pnl[t+2] should use signal[t]:")
print(f"    expected: signal[t] × ret = {_sig_trace.loc[t]:.0f} × {_r2:+.6f} = {_exp2:+.6f}")
print(f"    computed: {_act2:+.6f}")
print(f"    {'✅ MATCH' if np.isclose(_act2, _exp2) else '❌'}")

# Turnover check — |exec[t+1] - exec[t]| should be 1 for the linear ramp
_turn = _sig_trace.shift(2).fillna(0).diff().abs()
print(f"\n  Turnover on t+1 (trade at open[t+1]) = |signal[t-1] − signal[t-2]| = 1.0:")
print(f"    computed: {_turn.loc[u]:.1f}   {'✅' if np.isclose(_turn.loc[u], 1.0) else '❌'}")


## 2. Compute both PnL series and side-by-side stats

Each signal gets scored in BOTH modes — same signal, same sign, same shift, different returns.


In [ ]:
def pnl_of(sig, ret, cost):
    ex = sig.shift(2)
    return (ex*ret) - ex.fillna(0).diff().abs()*cost

def _sr(x):
    x=x.dropna(); return float(np.sqrt(252)*x.mean()/x.std()) if x.std()>0 else np.nan

def _max_dd_days(x):
    cum=x.cumsum(); dd=(cum-cum.cummax())<0
    if not dd.any(): return 0
    runs=dd.astype(int).groupby((dd!=dd.shift()).cumsum()).sum()
    return int(runs.max())

def _stats(pnl):
    a=pnl.dropna()
    if len(a)==0: return {"SR":np.nan}
    ann_ret=a.mean()*252; ann_vol=a.std()*np.sqrt(252)
    dd=a.cumsum()-a.cumsum().cummax()
    yr=a.groupby(a.index.year).apply(lambda x:_sr(x)).dropna()
    return {"SR":_sr(a), "ann_ret_%":ann_ret*100, "ann_vol_%":ann_vol*100,
            "max_dd_d":_max_dd_days(a),
            "pos_yr_%":(yr>0).mean()*100,
            "worst_yr":yr.min() if len(yr) else np.nan,
            "SR_of_SR":_sr(yr) if yr.std() else np.nan}

# Per-signal stats in both modes
rows = []
PNL_C2C = {}; PNL_O2O = {}
for n, sig in SIGNED.items():
    pnl_c = pnl_of(sig, ret_c2c, cost_c2c).loc[EVAL_START:EVAL_END]
    pnl_o = pnl_of(sig, ret_o2o, cost_o2o).loc[EVAL_START:EVAL_END]
    PNL_C2C[n] = pnl_c; PNL_O2O[n] = pnl_o
    sc = _stats(pnl_c); so = _stats(pnl_o)
    rows.append({
        "signal": n, "source": n.split("_")[0],
        "SR_c2c": round(sc["SR"],3), "SR_o2o": round(so["SR"],3),
        "SR_delta": round(so["SR"]-sc["SR"], 3),
        "ann_ret_c2c%": round(sc["ann_ret_%"],2), "ann_ret_o2o%": round(so["ann_ret_%"],2),
        "ann_vol_c2c%": round(sc["ann_vol_%"],2), "ann_vol_o2o%": round(so["ann_vol_%"],2),
        "max_dd_c2c": sc["max_dd_d"], "max_dd_o2o": so["max_dd_d"],
        "pos_yr_c2c%": round(sc["pos_yr_%"],1), "pos_yr_o2o%": round(so["pos_yr_%"],1),
        "SRoSR_c2c": round(sc["SR_of_SR"],2), "SRoSR_o2o": round(so["SR_of_SR"],2),
    })
side_by_side = pd.DataFrame(rows).set_index("signal")

print("=== Per-signal comparison — c2c vs o2o ===\n")
show1 = ["source","SR_c2c","SR_o2o","SR_delta","ann_ret_c2c%","ann_ret_o2o%",
         "max_dd_c2c","max_dd_o2o","pos_yr_c2c%","pos_yr_o2o%"]
display(side_by_side.sort_values("SR_delta", ascending=False)[show1])

# Winners / losers under o2o
print("\n=== Signals that IMPROVE most under o2o (Δ SR > 0) ===")
big_up = side_by_side[side_by_side["SR_delta"] > 0.05].sort_values("SR_delta", ascending=False)
display(big_up[show1])

print("\n=== Signals that DEGRADE most under o2o (Δ SR < 0) ===")
big_dn = side_by_side[side_by_side["SR_delta"] < -0.05].sort_values("SR_delta")
display(big_dn[show1])


## 3. Combined portfolio — c2c vs o2o


In [ ]:
SIG_DF = pd.DataFrame({n: SIGNED[n].reindex(ASSET.index) for n in SIGNED}).loc[EVAL_START:EVAL_END]

# Equal-weight ensemble in both modes
combo_c = pnl_of(SIG_DF.mean(axis=1), ret_c2c, cost_c2c).loc[EVAL_START:EVAL_END]
combo_o = pnl_of(SIG_DF.mean(axis=1), ret_o2o, cost_o2o).loc[EVAL_START:EVAL_END]

# Also: what if we PICK the best mode PER SIGNAL and combine?
best_mode = {n: ("o2o" if side_by_side.loc[n,"SR_o2o"] >= side_by_side.loc[n,"SR_c2c"] else "c2c")
             for n in SIG_DF.columns}
best_pnl_per_sig = pd.DataFrame({
    n: (PNL_O2O[n] if best_mode[n]=="o2o" else PNL_C2C[n]) for n in SIG_DF.columns
})
combo_best = best_pnl_per_sig.sum(axis=1) / len(SIG_DF.columns)

print("=== Combined portfolio (equal-weight across 9 survivors) ===\n")
combo_stats = pd.DataFrame({
    "c2c (default)":       _stats(combo_c),
    "o2o":                 _stats(combo_o),
    "best_mode_per_sig":   _stats(combo_best),
}).T.round(3)
display(combo_stats)

# 2026 YTD comparison
print("\n=== 2026 YTD by mode ===")
for label, p in [("c2c", combo_c), ("o2o", combo_o), ("best_mode", combo_best)]:
    ytd = p.loc["2026-01-01":]
    print(f"  {label:18s}  SR {_sr(ytd):+.3f}   ann ret {ytd.mean()*252*100:+.2f}%")

print(f"\n=== Which mode wins per signal ===")
for n in SIG_DF.columns:
    m = best_mode[n]
    marker = "▲ o2o" if m=="o2o" else "▽ c2c"
    print(f"  {marker}  {n:38s}  Δ SR = {side_by_side.loc[n,'SR_delta']:+.3f}")


## 4. Per-year SR heatmap — combined portfolio in both modes


In [ ]:
def per_year_sr(p):
    a=p.dropna()
    return a.groupby(a.index.year).apply(lambda x:_sr(x))

comp_yr = pd.DataFrame({
    "c2c":       per_year_sr(combo_c),
    "o2o":       per_year_sr(combo_o),
    "best_mode": per_year_sr(combo_best),
}).T
comp_yr = comp_yr.loc[:, comp_yr.columns >= 2013]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios":[2,3]})

# Top — combined per-year heatmap
ax = axes[0]
im = ax.imshow(comp_yr.values, cmap="RdYlGn", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(comp_yr.columns))); ax.set_xticklabels(comp_yr.columns)
ax.set_yticks(range(len(comp_yr))); ax.set_yticklabels(comp_yr.index, fontsize=10)
for i in range(comp_yr.shape[0]):
    for j in range(comp_yr.shape[1]):
        v = comp_yr.values[i,j]
        if np.isnan(v): continue
        ax.text(j,i,f"{v:+.1f}", ha="center",va="center", fontsize=8,
                color="white" if abs(v)>1.2 else "black")
ax.set_title("Combined portfolio — annual SR by mode")
plt.colorbar(im, ax=ax, shrink=0.8, label="annual SR")

# Bottom — cum PnL
ax = axes[1]
ax.plot(combo_c.cumsum()*100, color="#1565c0", lw=1.5, label=f"c2c (SR {_sr(combo_c):+.2f})")
ax.plot(combo_o.cumsum()*100, color="#e65100", lw=1.5, label=f"o2o (SR {_sr(combo_o):+.2f})")
ax.plot(combo_best.cumsum()*100, color="#2e7d32", lw=1.5,
         label=f"best-mode per sig (SR {_sr(combo_best):+.2f})")
ax.axhline(0, color="black", lw=0.4, ls="--", alpha=0.5)
ax.set_ylabel("cum PnL (%)")
ax.set_title(f"Combined equal-weight portfolio — {EVAL_START} → {EVAL_END}")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 5. Per-signal cum PnL comparison — where does each signal live?


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
for i, n in enumerate(SIGNED.keys()):
    ax = axes[i]
    p_c = PNL_C2C[n].cumsum()*100
    p_o = PNL_O2O[n].cumsum()*100
    winner = "o2o" if side_by_side.loc[n,"SR_o2o"] >= side_by_side.loc[n,"SR_c2c"] else "c2c"
    ax.plot(p_c.index, p_c.values, color="#1565c0", lw=1.1,
             label=f"c2c SR {side_by_side.loc[n,'SR_c2c']:+.2f}")
    ax.plot(p_o.index, p_o.values, color="#e65100", lw=1.1,
             label=f"o2o SR {side_by_side.loc[n,'SR_o2o']:+.2f}")
    ax.axhline(0, color="black", lw=0.3, ls="--", alpha=0.5)
    ax.set_title(f"{n[:34]}   ← {winner}", fontsize=8)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()


## 6. Bottom line

The three questions to answer from the tables + plots above:

1. **Does the ensemble Sharpe improve under o2o?** — compare `c2c` row to `o2o` row in § 3.
2. **Which specific signals show the biggest o2o improvement?** — top of the `SR_delta` sorted table in § 2.
3. **Should we switch modes per signal, or stick with one?** — compare `best_mode_per_sig` vs plain `o2o` / `c2c`.

Two operational considerations before switching:
- **Execution feasibility**: an o2o strategy requires a market-on-open order routed reliably to TAIFEX at 08:45 TPE. If your execution stack doesn't do MOO, o2o SR is theoretical.
- **Cost model**: I used the OPEN price as the reference for cost_o2o (turnover happens at open in o2o mode). If your real execution has different slippage at the open vs. VWAP-into-close, the o2o cost picture might be worse (or better) than modeled.


## 7. Full `cta.Simulate` dashboard for the ensemble + long-tilt sweep

Two questions rolled into one section:

**A. What does the plain ensemble look like on the standard dashboard, and what's its beta to MTX buy-and-hold?**

The equal-weight ensemble of the 9 signed survivors is passed to `cta.Simulate` — this runs the standard multi-panel diagnostic (cum PnL vs. buy-and-hold, drawdown, PnL histogram, per-year, monthly heatmap, β/α decomposition, expiry CASR, TSMC CASR, weekday breakdown, front-vs-back, vol-matched PnL). The (1,0) panel reports the c2c β directly. I also compute o2o β and c2c/o2o SR alongside for reference.

**B. Long-tilt sweep — blend the ensemble with buy-and-hold**

Operation: `tilted[k] = tanh(raw_ensemble + k)` where `raw_ensemble = mean of 9 SIGNED signals` (pre-tanh, roughly in [−1, +1]).

| k value | signal behavior |
|---|---|
| **k = 0** | pure ensemble — centered around 0, both long and short |
| k = +0.5 | mostly long — negative reduced, moderate short still possible |
| k = +1   | strongly long-tilted — signal rarely goes short |
| k = +2   | ≈ always long, position size ≈ +0.96 to +1 (nearly buy-and-hold) |
| k = +5   | fully saturated at +1 (indistinguishable from buy-and-hold) |
| k = −1   | strongly short-tilted (for comparison / testing symmetry) |

For each k we report: SR (c2c AND o2o), β, α/yr, ann_ret, ann_vol, max_dd. The goal: find the mix that trades off pure-alpha (k=0) against reduced-drawdown-plus-market-exposure (k>0).


In [ ]:
# ── § 7.1  Compute the ensemble signal + β diagnostic ─────────────────────

# Raw ensemble: mean of the 9 SIGNED signals BEFORE final tanh (needed for
# the tilt sweep to work on the additive scale)
raw_ensemble = SIG_DF.mean(axis=1).rename("raw_ensemble")
ensemble     = np.tanh(raw_ensemble).rename("ensemble_tanh")

# β / α via cta.batch_signal_stats (c2c convention, matches Simulate dashboard)
tbl_ens = cta.batch_signal_stats(
    {"ensemble": ensemble},
    ASSET, start=EVAL_START, end=EVAL_END, auto_flip=False,
)
print("=== Ensemble diagnostic (c2c basis — matches cta.Simulate) ===")
show = ["SR_gross","SR_net","alpha_ann_pct","beta","corr_bh","max_dd_pct","max_dd_days","calmar",
        "positive_years","n_years","SR_of_SR"]
display(tbl_ens[show].round(3))

# Also o2o SR for reference
_pnl_o2o = pnl_of(ensemble, ret_o2o, cost_o2o).loc[EVAL_START:EVAL_END]
_pnl_c2c = pnl_of(ensemble, ret_c2c, cost_c2c).loc[EVAL_START:EVAL_END]
print(f"\n  o2o SR : {_sr(_pnl_o2o):+.3f}")
print(f"  c2c SR : {_sr(_pnl_c2c):+.3f}")
print(f"  β to MTX (c2c): {float(tbl_ens['beta'].iloc[0]):+.3f}    "
      f"corr with buy-and-hold: {float(tbl_ens['corr_bh'].iloc[0]):+.3f}")


In [ ]:
# ── § 7.2  cta.Simulate — full diagnostic dashboard on the ensemble ────────
# (uses c2c convention — this is the standard dashboard people expect)
_ = cta.Simulate(ensemble, asset="mtx", time_granularity="1d",
                 normalize=False, start_date=EVAL_START, end_date=EVAL_END)


In [ ]:
# ── § 7.3  Long-tilt sweep — tanh(raw_ensemble + k) for various k ─────────

K_VALUES = [-1.0, -0.5, 0.0, 0.25, 0.5, 1.0, 2.0, 5.0]

def _stats_full(pnl):
    a = pnl.dropna()
    if len(a) == 0: return {}
    ann_ret = a.mean() * 252; ann_vol = a.std() * np.sqrt(252)
    dd = a.cumsum() - a.cumsum().cummax()
    yr = a.groupby(a.index.year).apply(lambda x: _sr(x)).dropna()
    return {
        "SR":         round(_sr(a), 3),
        "ann_ret_%":  round(ann_ret * 100, 2),
        "ann_vol_%":  round(ann_vol * 100, 2),
        "max_dd_d":   _max_dd_days(a),
        "pos_yr_%":   round((yr > 0).mean() * 100, 1),
        "worst_yr":   round(yr.min(), 2) if len(yr) else np.nan,
    }

rows = []
tilt_pnl_c = {}; tilt_pnl_o = {}
for k in K_VALUES:
    sig_k = np.tanh(raw_ensemble + k).rename(f"tilt_k={k:+.2f}")
    p_c = pnl_of(sig_k, ret_c2c, cost_c2c).loc[EVAL_START:EVAL_END]
    p_o = pnl_of(sig_k, ret_o2o, cost_o2o).loc[EVAL_START:EVAL_END]
    tilt_pnl_c[k] = p_c; tilt_pnl_o[k] = p_o
    # β via cta.batch_signal_stats (c2c)
    tbl_k = cta.batch_signal_stats({"x": sig_k}, ASSET,
                                    start=EVAL_START, end=EVAL_END, auto_flip=False)
    beta_c  = float(tbl_k["beta"].iloc[0])
    alpha_c = float(tbl_k["alpha_ann_pct"].iloc[0])
    sc = _stats_full(p_c); so = _stats_full(p_o)
    # Average absolute position (rough long-tilt magnitude proxy)
    avg_pos = float(sig_k.mean())
    rows.append({
        "k":        k,
        "avg_pos":  round(avg_pos, 3),
        "beta":     round(beta_c, 3),
        "alpha_%":  round(alpha_c, 2),
        "SR_c2c":   sc.get("SR", np.nan),
        "SR_o2o":   so.get("SR", np.nan),
        "ret_c2c_%":sc.get("ann_ret_%", np.nan),
        "ret_o2o_%":so.get("ann_ret_%", np.nan),
        "vol_c2c_%":sc.get("ann_vol_%", np.nan),
        "vol_o2o_%":so.get("ann_vol_%", np.nan),
        "max_dd_o2o":so.get("max_dd_d", np.nan),
        "pos_yr_o2o":so.get("pos_yr_%", np.nan),
        "worst_yr_o2o":so.get("worst_yr", np.nan),
    })

# Also add BUY-AND-HOLD as the k=∞ reference
bh_sig = pd.Series(1.0, index=ASSET.index).rename("buy_hold")
bh_pc = pnl_of(bh_sig, ret_c2c, cost_c2c).loc[EVAL_START:EVAL_END]
bh_po = pnl_of(bh_sig, ret_o2o, cost_o2o).loc[EVAL_START:EVAL_END]
bh_tbl = cta.batch_signal_stats({"x": bh_sig}, ASSET, start=EVAL_START, end=EVAL_END, auto_flip=False)
bh_sc = _stats_full(bh_pc); bh_so = _stats_full(bh_po)
rows.append({
    "k": "∞ (BH)", "avg_pos": 1.0,
    "beta": round(float(bh_tbl["beta"].iloc[0]), 3),
    "alpha_%": round(float(bh_tbl["alpha_ann_pct"].iloc[0]), 2),
    "SR_c2c": bh_sc.get("SR"), "SR_o2o": bh_so.get("SR"),
    "ret_c2c_%": bh_sc.get("ann_ret_%"), "ret_o2o_%": bh_so.get("ann_ret_%"),
    "vol_c2c_%": bh_sc.get("ann_vol_%"), "vol_o2o_%": bh_so.get("ann_vol_%"),
    "max_dd_o2o": bh_so.get("max_dd_d"), "pos_yr_o2o": bh_so.get("pos_yr_%"),
    "worst_yr_o2o": bh_so.get("worst_yr"),
})
tilt_df = pd.DataFrame(rows).set_index("k")

print("=== Tilt sweep — tanh(raw_ensemble + k) ===\n")
print("(Each row: β / α from cta.batch_signal_stats; SR/ret/vol/dd computed with cost)\n")
display(tilt_df.round(3))


In [ ]:
# ── § 7.4  Tilt sweep — visual comparison (cum PnL + β vs SR frontier) ─────

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios":[3, 2]})

# Top — o2o cum PnLs for each k
ax = axes[0]
cmap = plt.get_cmap("coolwarm")
n_k = len(K_VALUES)
for i, k in enumerate(K_VALUES):
    color = cmap(i / max(1, n_k - 1))
    lbl = f"k={k:+.2f}  β={tilt_df.loc[k,'beta']:+.2f}  SR={tilt_df.loc[k,'SR_o2o']:+.2f}"
    ax.plot(tilt_pnl_o[k].cumsum() * 100, color=color, lw=1.4, label=lbl)
# Reference: pure buy-and-hold
ax.plot(bh_po.cumsum() * 100, color="black", lw=1.2, ls="--", alpha=0.75,
         label=f"buy-and-hold  β=+1  SR={tilt_df.loc['∞ (BH)','SR_o2o']:+.2f}")
ax.axhline(0, color="black", lw=0.3, ls="--", alpha=0.4)
ax.set_ylabel("cum PnL (% of $1 signal)")
ax.set_title(f"Long-tilt sweep — o2o cum PnL  ({EVAL_START} → {EVAL_END})")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

# Bottom — β vs SR frontier + risk-metrics twin panel
ax = axes[1]
# Use numeric x (average position)
xs   = tilt_df["avg_pos"].astype(float).values
srs  = tilt_df["SR_o2o"].values
bts  = tilt_df["beta"].values
labels = [str(k) for k in tilt_df.index]

# Scatter SR vs avg_pos, sized by (1 - β) so pure-alpha (low β) stands out
sizes = np.clip((1.5 - np.abs(bts)) * 200, 30, 500)
sc = ax.scatter(xs, srs, s=sizes, c=bts, cmap="coolwarm",
                vmin=-1, vmax=1, edgecolor="black", alpha=0.9)
for x, y, lbl in zip(xs, srs, labels):
    ax.annotate(lbl, (x, y), fontsize=8, xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("average signal position (long-tilt magnitude)")
ax.set_ylabel("SR (o2o)")
ax.axhline(tilt_df.loc[0.0, "SR_o2o"], color="#1565c0", lw=0.5, ls="--", alpha=0.6,
           label=f"pure ensemble (k=0) SR = {tilt_df.loc[0.0,'SR_o2o']:+.2f}")
ax.axhline(tilt_df.loc["∞ (BH)", "SR_o2o"], color="black", lw=0.5, ls="--", alpha=0.6,
           label=f"buy-and-hold SR = {tilt_df.loc['∞ (BH)','SR_o2o']:+.2f}")
plt.colorbar(sc, ax=ax, label="β to MTX (colour)")
ax.set_title("Frontier — SR vs long-tilt (dot size = alpha-purity, colour = β)")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# ── § 7.5  Which k is optimal? ─────────────────────────────────────────────

best_o2o = tilt_df["SR_o2o"].idxmax()
best_c2c = tilt_df["SR_c2c"].idxmax()

print(f"=== Tilt-sweep summary ===")
print(f"  best SR o2o  : k = {best_o2o}   (SR = {tilt_df.loc[best_o2o,'SR_o2o']:+.3f}, "
      f"β = {tilt_df.loc[best_o2o,'beta']:+.3f})")
print(f"  best SR c2c  : k = {best_c2c}   (SR = {tilt_df.loc[best_c2c,'SR_c2c']:+.3f}, "
      f"β = {tilt_df.loc[best_c2c,'beta']:+.3f})")

# Interpret
print(f"\nInterpretation:")
print(f"  pure ensemble (k=0): SR o2o = {tilt_df.loc[0.0,'SR_o2o']:+.3f}, β = {tilt_df.loc[0.0,'beta']:+.3f}")
print(f"  buy-and-hold      : SR o2o = {tilt_df.loc['∞ (BH)','SR_o2o']:+.3f}, β = +1")
print(f"  best tilt         : k = {best_o2o}, adds β = {tilt_df.loc[best_o2o,'beta']:+.3f} of market exposure")

if isinstance(best_o2o, (int, float)) and best_o2o > 0:
    delta_sr = tilt_df.loc[best_o2o,'SR_o2o'] - tilt_df.loc[0.0,'SR_o2o']
    if delta_sr > 0.05:
        print(f"\n  ✅ Adding long-tilt (k={best_o2o}) IMPROVES SR by {delta_sr:+.3f}"
              f" — the ensemble benefits from riding some market beta.")
    else:
        print(f"\n  ▽ Long-tilt doesn't help much (Δ SR = {delta_sr:+.3f})"
              f" — the ensemble already captures the good moves without needing market exposure.")
